In [1]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


In [3]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import polars as pl

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

from xgboost import XGBRegressor

print("Imports ready")

Imports ready


In [4]:
PROJECT_ROOT = Path.cwd().parent

TRAIN_FILE = PROJECT_ROOT / "data" / "splits" / "train" / "trips_train.parquet"
VALIDATION_FILE = PROJECT_ROOT / "data" / "splits" / "validation" / "trips_validation.parquet"

print("Train exists:", TRAIN_FILE.exists())
print("Validation exists:", VALIDATION_FILE.exists())

Train exists: True
Validation exists: True


In [5]:
train_df = pl.read_parquet(TRAIN_FILE)
validation_df = pl.read_parquet(VALIDATION_FILE)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (33792312, 20)
Validation shape: (6730119, 20)


In [6]:
train_df = train_df.filter(
    (pl.col("base_fare").is_not_null()) &
    (pl.col("base_fare") <= 5000)
)

print("Filtered train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Filtered train shape: (33792310, 20)
Validation shape: (6730119, 20)


In [8]:
def add_time_features(df):
    return (
        df
        .with_columns(
            pl.col("pickup_timestamp").dt.hour().alias("pickup_hour"),
            pl.col("pickup_timestamp").dt.weekday().alias("pickup_day_of_week"),
            pl.col("pickup_timestamp").dt.month().alias("pickup_month"),
        )
        .with_columns(
            (pl.col("pickup_day_of_week") >= 6)
            .cast(pl.Int8)
            .alias("is_weekend")
        )
    )

train_df = add_time_features(train_df)
validation_df = add_time_features(validation_df)

FEATURES = [
    "provider_code",
    "rider_count",
    "rate_class_id",
    "origin_loc_id",
    "dest_loc_id",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_month",
    "is_weekend",
]

print("Features:", FEATURES)

Features: ['provider_code', 'rider_count', 'rate_class_id', 'origin_loc_id', 'dest_loc_id', 'pickup_hour', 'pickup_day_of_week', 'pickup_month', 'is_weekend']


In [9]:
rider_median = train_df.select(
    pl.col("rider_count").median()
).item()

rate_class_mode = train_df.select(
    pl.col("rate_class_id").mode().first()
).item()

print("Rider count median:", rider_median)
print("Rate class mode:", rate_class_mode)

Rider count median: 1.0
Rate class mode: 1


In [10]:
train_model_df = train_df.with_columns(
    pl.col("rider_count").fill_null(rider_median),
    pl.col("rate_class_id").fill_null(rate_class_mode),
)

validation_model_df = validation_df.with_columns(
    pl.col("rider_count").fill_null(rider_median),
    pl.col("rate_class_id").fill_null(rate_class_mode),
)

X_train_np = (
    train_model_df
    .select(FEATURES)
    .to_numpy()
    .astype(np.float32)
)

y_train_np = (
    train_model_df
    .select("base_fare")
    .to_numpy()
    .ravel()
    .astype(np.float32)
)

X_validation_np = (
    validation_model_df
    .select(FEATURES)
    .to_numpy()
    .astype(np.float32)
)

y_validation_np = (
    validation_model_df
    .select("base_fare")
    .to_numpy()
    .ravel()
    .astype(np.float32)
)

print("X_train:", X_train_np.shape, X_train_np.dtype)
print("y_train:", y_train_np.shape, y_train_np.dtype)
print("X_validation:", X_validation_np.shape, X_validation_np.dtype)
print("y_validation:", y_validation_np.shape, y_validation_np.dtype)

X_train: (33792310, 9) float32
y_train: (33792310,) float32
X_validation: (6730119, 9) float32
y_validation: (6730119,) float32


In [11]:
from xgboost import XGBRegressor

xgb_run_1 = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)

start_time = time.time()

xgb_run_1.fit(
    X_train_np,
    y_train_np
)

xgb_run_1_time = time.time() - start_time

xgb_run_1_predictions = xgb_run_1.predict(
    X_validation_np
)

xgb_run_1_mae = mean_absolute_error(
    y_validation_np,
    xgb_run_1_predictions
)

xgb_run_1_rmse = root_mean_squared_error(
    y_validation_np,
    xgb_run_1_predictions
)

xgb_run_1_r2 = r2_score(
    y_validation_np,
    xgb_run_1_predictions
)

print("XGBoost - Run 1")
print("n_estimators = 100, learning_rate = 0.1, max_depth = 6")
print("MAE:", round(xgb_run_1_mae, 4))
print("RMSE:", round(xgb_run_1_rmse, 4))
print("R²:", round(xgb_run_1_r2, 4))
print("Training time (seconds):", round(xgb_run_1_time, 2))

XGBoost - Run 1
n_estimators = 100, learning_rate = 0.1, max_depth = 6
MAE: 7.2767
RMSE: 11.7592
R²: 0.5511
Training time (seconds): 138.34


In [12]:
xgb_run_2 = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)

start_time = time.time()

xgb_run_2.fit(
    X_train_np,
    y_train_np
)

xgb_run_2_time = time.time() - start_time

xgb_run_2_predictions = xgb_run_2.predict(
    X_validation_np
)

xgb_run_2_mae = mean_absolute_error(
    y_validation_np,
    xgb_run_2_predictions
)

xgb_run_2_rmse = root_mean_squared_error(
    y_validation_np,
    xgb_run_2_predictions
)

xgb_run_2_r2 = r2_score(
    y_validation_np,
    xgb_run_2_predictions
)

print("XGBoost - Run 2")
print("n_estimators = 200, learning_rate = 0.05, max_depth = 8")
print("MAE:", round(xgb_run_2_mae, 4))
print("RMSE:", round(xgb_run_2_rmse, 4))
print("R²:", round(xgb_run_2_r2, 4))
print("Training time (seconds):", round(xgb_run_2_time, 2))

XGBoost - Run 2
n_estimators = 200, learning_rate = 0.05, max_depth = 8
MAE: 6.6752
RMSE: 11.1574
R²: 0.5958
Training time (seconds): 426.66


In [13]:
xgb_run_3 = XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=10,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)

start_time = time.time()

xgb_run_3.fit(
    X_train_np,
    y_train_np
)

xgb_run_3_time = time.time() - start_time

xgb_run_3_predictions = xgb_run_3.predict(
    X_validation_np
)

xgb_run_3_mae = mean_absolute_error(
    y_validation_np,
    xgb_run_3_predictions
)

xgb_run_3_rmse = root_mean_squared_error(
    y_validation_np,
    xgb_run_3_predictions
)

xgb_run_3_r2 = r2_score(
    y_validation_np,
    xgb_run_3_predictions
)

print("XGBoost - Run 3")
print("n_estimators = 300, learning_rate = 0.1, max_depth = 10")
print("MAE:", round(xgb_run_3_mae, 4))
print("RMSE:", round(xgb_run_3_rmse, 4))
print("R²:", round(xgb_run_3_r2, 4))
print("Training time (seconds):", round(xgb_run_3_time, 2))

XGBoost - Run 3
n_estimators = 300, learning_rate = 0.1, max_depth = 10
MAE: 5.2836
RMSE: 9.8081
R²: 0.6877
Training time (seconds): 706.93


In [14]:
xgb_run_4 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=12,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
)

start_time = time.time()

xgb_run_4.fit(
    X_train_np,
    y_train_np
)

xgb_run_4_time = time.time() - start_time

xgb_run_4_predictions = xgb_run_4.predict(
    X_validation_np
)

xgb_run_4_mae = mean_absolute_error(
    y_validation_np,
    xgb_run_4_predictions
)

xgb_run_4_rmse = root_mean_squared_error(
    y_validation_np,
    xgb_run_4_predictions
)

xgb_run_4_r2 = r2_score(
    y_validation_np,
    xgb_run_4_predictions
)

print("XGBoost - Run 4")
print("n_estimators = 500, learning_rate = 0.05, max_depth = 12")
print("MAE:", round(xgb_run_4_mae, 4))
print("RMSE:", round(xgb_run_4_rmse, 4))
print("R²:", round(xgb_run_4_r2, 4))
print("Training time (seconds):", round(xgb_run_4_time, 2))

XGBoost - Run 4
n_estimators = 500, learning_rate = 0.05, max_depth = 12
MAE: 5.052
RMSE: 9.5788
R²: 0.7021
Training time (seconds): 1391.93


In [15]:
xgb_tuning_results = pd.DataFrame([
    {
        "Run": 1,
        "n_estimators": 100,
        "learning_rate": 0.10,
        "max_depth": 6,
        "MAE": xgb_run_1_mae,
        "RMSE": xgb_run_1_rmse,
        "R2": xgb_run_1_r2,
        "Training_Time_Seconds": xgb_run_1_time,
    },
    {
        "Run": 2,
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 8,
        "MAE": xgb_run_2_mae,
        "RMSE": xgb_run_2_rmse,
        "R2": xgb_run_2_r2,
        "Training_Time_Seconds": xgb_run_2_time,
    },
    {
        "Run": 3,
        "n_estimators": 300,
        "learning_rate": 0.10,
        "max_depth": 10,
        "MAE": xgb_run_3_mae,
        "RMSE": xgb_run_3_rmse,
        "R2": xgb_run_3_r2,
        "Training_Time_Seconds": xgb_run_3_time,
    },
    {
        "Run": 4,
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 12,
        "MAE": xgb_run_4_mae,
        "RMSE": xgb_run_4_rmse,
        "R2": xgb_run_4_r2,
        "Training_Time_Seconds": xgb_run_4_time,
    },
])

xgb_tuning_results.round(4)

,Run,n_estimators,learning_rate,max_depth,MAE,RMSE,R2,Training_Time_Seconds
0,1,100,0.10,6,7.2767,11.7592,0.5511,138.3416
1,2,200,0.05,8,6.6752,11.1574,0.5958,426.6611
2,3,300,0.10,10,5.2836,9.8081,0.6877,706.9315
3,4,500,0.05,12,5.0520,9.5788,0.7021,1391.9337


In [17]:
import catboost

print("CatBoost version:", catboost.__version__)

CatBoost version: 1.2.10


In [18]:
from catboost import CatBoostRegressor

cat_run_1 = CatBoostRegressor(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)

start_time = time.time()

cat_run_1.fit(
    X_train_np,
    y_train_np
)

cat_run_1_time = time.time() - start_time

cat_run_1_predictions = cat_run_1.predict(
    X_validation_np
)

cat_run_1_mae = mean_absolute_error(
    y_validation_np,
    cat_run_1_predictions
)

cat_run_1_rmse = root_mean_squared_error(
    y_validation_np,
    cat_run_1_predictions
)

cat_run_1_r2 = r2_score(
    y_validation_np,
    cat_run_1_predictions
)

print("CatBoost - Run 1")
print("iterations = 200, learning_rate = 0.1, depth = 6")
print("MAE:", round(cat_run_1_mae, 4))
print("RMSE:", round(cat_run_1_rmse, 4))
print("R²:", round(cat_run_1_r2, 4))
print("Training time (seconds):", round(cat_run_1_time, 2))

CatBoost - Run 1
iterations = 200, learning_rate = 0.1, depth = 6
MAE: 7.6116
RMSE: 12.0886
R²: 0.5255
Training time (seconds): 552.4


In [19]:
cat_run_2 = CatBoostRegressor(
    iterations=400,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)

start_time = time.time()

cat_run_2.fit(
    X_train_np,
    y_train_np
)

cat_run_2_time = time.time() - start_time

cat_run_2_predictions = cat_run_2.predict(
    X_validation_np
)

cat_run_2_mae = mean_absolute_error(
    y_validation_np,
    cat_run_2_predictions
)

cat_run_2_rmse = root_mean_squared_error(
    y_validation_np,
    cat_run_2_predictions
)

cat_run_2_r2 = r2_score(
    y_validation_np,
    cat_run_2_predictions
)

print("CatBoost - Run 2")
print("iterations = 400, learning_rate = 0.05, depth = 8")
print("MAE:", round(cat_run_2_mae, 4))
print("RMSE:", round(cat_run_2_rmse, 4))
print("R²:", round(cat_run_2_r2, 4))
print("Training time (seconds):", round(cat_run_2_time, 2))

CatBoost - Run 2
iterations = 400, learning_rate = 0.05, depth = 8
MAE: 7.3238
RMSE: 11.7871
R²: 0.5489
Training time (seconds): 1255.3


In [22]:
cat_run_3 = CatBoostRegressor(
    iterations=500,
    learning_rate=0.08,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
    thread_count=-1,
)

start_time = time.time()

cat_run_3.fit(
    X_train_np,
    y_train_np
)

cat_run_3_time = time.time() - start_time

cat_run_3_predictions = cat_run_3.predict(
    X_validation_np
)

cat_run_3_mae = mean_absolute_error(
    y_validation_np,
    cat_run_3_predictions
)

cat_run_3_rmse = root_mean_squared_error(
    y_validation_np,
    cat_run_3_predictions
)

cat_run_3_r2 = r2_score(
    y_validation_np,
    cat_run_3_predictions
)

print("CatBoost - Run 3")
print("iterations = 500, learning_rate = 0.08, depth = 8")
print("MAE:", round(cat_run_3_mae, 4))
print("RMSE:", round(cat_run_3_rmse, 4))
print("R²:", round(cat_run_3_r2, 4))
print("Training time (seconds):", round(cat_run_3_time, 2))

CatBoostError: bad allocation

In [23]:
CATBOOST_SAMPLE_SIZE = 5_000_000

rng = np.random.default_rng(42)

sample_idx = rng.choice(
    X_train_np.shape[0],
    size=CATBOOST_SAMPLE_SIZE,
    replace=False
)

X_train_cat = X_train_np[sample_idx]
y_train_cat = y_train_np[sample_idx]

print("CatBoost training sample:", X_train_cat.shape)
print("CatBoost target sample:", y_train_cat.shape)
print("Validation remains full:", X_validation_np.shape)

CatBoost training sample: (5000000, 9)
CatBoost target sample: (5000000,)
Validation remains full: (6730119, 9)


In [24]:
import gc

for name in [
    "xgb_run_1", "xgb_run_2", "xgb_run_3", "xgb_run_4",
    "xgb_run_1_predictions", "xgb_run_2_predictions",
    "xgb_run_3_predictions", "xgb_run_4_predictions",
    "cat_run_1", "cat_run_2", "cat_run_3",
    "cat_run_1_predictions", "cat_run_2_predictions"
]:
    if name in globals():
        del globals()[name]

gc.collect()

print("Old model objects cleared")

Old model objects cleared


In [25]:
cat_run_1 = CatBoostRegressor(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)

start_time = time.time()

cat_run_1.fit(
    X_train_cat,
    y_train_cat
)

cat_run_1_time = time.time() - start_time

cat_run_1_predictions = cat_run_1.predict(
    X_validation_np
)

cat_run_1_mae = mean_absolute_error(
    y_validation_np,
    cat_run_1_predictions
)

cat_run_1_rmse = root_mean_squared_error(
    y_validation_np,
    cat_run_1_predictions
)

cat_run_1_r2 = r2_score(
    y_validation_np,
    cat_run_1_predictions
)

print("CatBoost - Run 1")
print("Training rows = 5,000,000")
print("iterations = 200, learning_rate = 0.1, depth = 6")
print("MAE:", round(cat_run_1_mae, 4))
print("RMSE:", round(cat_run_1_rmse, 4))
print("R²:", round(cat_run_1_r2, 4))
print("Training time (seconds):", round(cat_run_1_time, 2))

CatBoost - Run 1
Training rows = 5,000,000
iterations = 200, learning_rate = 0.1, depth = 6
MAE: 7.6288
RMSE: 12.1123
R²: 0.5237
Training time (seconds): 86.34


In [26]:
cat_run_2 = CatBoostRegressor(
    iterations=400,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)

start_time = time.time()

cat_run_2.fit(
    X_train_cat,
    y_train_cat
)

cat_run_2_time = time.time() - start_time

cat_run_2_predictions = cat_run_2.predict(
    X_validation_np
)

cat_run_2_mae = mean_absolute_error(
    y_validation_np,
    cat_run_2_predictions
)

cat_run_2_rmse = root_mean_squared_error(
    y_validation_np,
    cat_run_2_predictions
)

cat_run_2_r2 = r2_score(
    y_validation_np,
    cat_run_2_predictions
)

print("CatBoost - Run 2")
print("Training rows = 5,000,000")
print("iterations = 400, learning_rate = 0.05, depth = 8")
print("MAE:", round(cat_run_2_mae, 4))
print("RMSE:", round(cat_run_2_rmse, 4))
print("R²:", round(cat_run_2_r2, 4))
print("Training time (seconds):", round(cat_run_2_time, 2))

CatBoost - Run 2
Training rows = 5,000,000
iterations = 400, learning_rate = 0.05, depth = 8
MAE: 7.3324
RMSE: 11.7827
R²: 0.5493
Training time (seconds): 202.92


In [27]:
cat_run_3 = CatBoostRegressor(
    iterations=600,
    learning_rate=0.08,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)

start_time = time.time()

cat_run_3.fit(
    X_train_cat,
    y_train_cat
)

cat_run_3_time = time.time() - start_time

cat_run_3_predictions = cat_run_3.predict(
    X_validation_np
)

cat_run_3_mae = mean_absolute_error(
    y_validation_np,
    cat_run_3_predictions
)

cat_run_3_rmse = root_mean_squared_error(
    y_validation_np,
    cat_run_3_predictions
)

cat_run_3_r2 = r2_score(
    y_validation_np,
    cat_run_3_predictions
)

print("CatBoost - Run 3")
print("Training rows = 5,000,000")
print("iterations = 600, learning_rate = 0.08, depth = 8")
print("MAE:", round(cat_run_3_mae, 4))
print("RMSE:", round(cat_run_3_rmse, 4))
print("R²:", round(cat_run_3_r2, 4))
print("Training time (seconds):", round(cat_run_3_time, 2))

CatBoost - Run 3
Training rows = 5,000,000
iterations = 600, learning_rate = 0.08, depth = 8
MAE: 6.7013
RMSE: 11.1757
R²: 0.5945
Training time (seconds): 304.34


In [28]:
cat_run_4 = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=10,
    loss_function="RMSE",
    random_seed=42,
    verbose=0,
)

start_time = time.time()

cat_run_4.fit(
    X_train_cat,
    y_train_cat
)

cat_run_4_time = time.time() - start_time

cat_run_4_predictions = cat_run_4.predict(
    X_validation_np
)

cat_run_4_mae = mean_absolute_error(
    y_validation_np,
    cat_run_4_predictions
)

cat_run_4_rmse = root_mean_squared_error(
    y_validation_np,
    cat_run_4_predictions
)

cat_run_4_r2 = r2_score(
    y_validation_np,
    cat_run_4_predictions
)

print("CatBoost - Run 4")
print("Training rows = 5,000,000")
print("iterations = 800, learning_rate = 0.05, depth = 10")
print("MAE:", round(cat_run_4_mae, 4))
print("RMSE:", round(cat_run_4_rmse, 4))
print("R²:", round(cat_run_4_r2, 4))
print("Training time (seconds):", round(cat_run_4_time, 2))

CatBoost - Run 4
Training rows = 5,000,000
iterations = 800, learning_rate = 0.05, depth = 10
MAE: 6.4822
RMSE: 10.9731
R²: 0.6091
Training time (seconds): 816.94


In [29]:
catboost_results = pd.DataFrame([
    {
        "Run": 1,
        "Training_Rows": 5_000_000,
        "Iterations": 200,
        "Learning_Rate": 0.10,
        "Depth": 6,
        "MAE": 7.6288,
        "RMSE": 12.1123,
        "R2": 0.5237,
        "Training_Time_Seconds": 86.34,
    },
    {
        "Run": 2,
        "Training_Rows": 5_000_000,
        "Iterations": 400,
        "Learning_Rate": 0.05,
        "Depth": 8,
        "MAE": 7.3324,
        "RMSE": 11.7827,
        "R2": 0.5493,
        "Training_Time_Seconds": 202.92,
    },
    {
        "Run": 3,
        "Training_Rows": 5_000_000,
        "Iterations": 600,
        "Learning_Rate": 0.08,
        "Depth": 8,
        "MAE": 6.7013,
        "RMSE": 11.1757,
        "R2": 0.5945,
        "Training_Time_Seconds": 304.34,
    },
    {
        "Run": 4,
        "Training_Rows": 5_000_000,
        "Iterations": 800,
        "Learning_Rate": 0.05,
        "Depth": 10,
        "MAE": 6.4822,
        "RMSE": 10.9731,
        "R2": 0.6091,
        "Training_Time_Seconds": 816.94,
    },
])

catboost_results

,Run,Training_Rows,Iterations,Learning_Rate,Depth,MAE,RMSE,R2,Training_Time_Seconds
0,1,5000000,200,0.10,6,7.6288,12.1123,0.5237,86.34
1,2,5000000,400,0.05,8,7.3324,11.7827,0.5493,202.92
2,3,5000000,600,0.08,8,6.7013,11.1757,0.5945,304.34
3,4,5000000,800,0.05,10,6.4822,10.9731,0.6091,816.94


In [30]:
from sklearn.ensemble import RandomForestRegressor

rf_run_1 = RandomForestRegressor(
    n_estimators=50,
    max_depth=15,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
)

start_time = time.time()

rf_run_1.fit(
    X_train_cat,
    y_train_cat
)

rf_run_1_time = time.time() - start_time

rf_run_1_predictions = rf_run_1.predict(
    X_validation_np
)

rf_run_1_mae = mean_absolute_error(
    y_validation_np,
    rf_run_1_predictions
)

rf_run_1_rmse = root_mean_squared_error(
    y_validation_np,
    rf_run_1_predictions
)

rf_run_1_r2 = r2_score(
    y_validation_np,
    rf_run_1_predictions
)

print("Random Forest - Run 1")
print("Training rows = 5,000,000")
print("n_estimators = 50, max_depth = 15, min_samples_leaf = 50")
print("MAE:", round(rf_run_1_mae, 4))
print("RMSE:", round(rf_run_1_rmse, 4))
print("R²:", round(rf_run_1_r2, 4))
print("Training time (seconds):", round(rf_run_1_time, 2))

Random Forest - Run 1
Training rows = 5,000,000
n_estimators = 50, max_depth = 15, min_samples_leaf = 50
MAE: 6.2394
RMSE: 10.8024
R²: 0.6211
Training time (seconds): 533.5


In [31]:
rf_run_2 = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=50,
    random_state=42,
    n_jobs=-1,
)

start_time = time.time()

rf_run_2.fit(
    X_train_cat,
    y_train_cat
)

rf_run_2_time = time.time() - start_time

rf_run_2_predictions = rf_run_2.predict(
    X_validation_np
)

rf_run_2_mae = mean_absolute_error(
    y_validation_np,
    rf_run_2_predictions
)

rf_run_2_rmse = root_mean_squared_error(
    y_validation_np,
    rf_run_2_predictions
)

rf_run_2_r2 = r2_score(
    y_validation_np,
    rf_run_2_predictions
)

print("Random Forest - Run 2")
print("Training rows = 5,000,000")
print("n_estimators = 100, max_depth = 20, min_samples_leaf = 50")
print("MAE:", round(rf_run_2_mae, 4))
print("RMSE:", round(rf_run_2_rmse, 4))
print("R²:", round(rf_run_2_r2, 4))
print("Training time (seconds):", round(rf_run_2_time, 2))

Random Forest - Run 2
Training rows = 5,000,000
n_estimators = 100, max_depth = 20, min_samples_leaf = 50
MAE: 5.2802
RMSE: 9.8639
R²: 0.6841
Training time (seconds): 1086.7


In [32]:
rf_run_3 = RandomForestRegressor(
    n_estimators=150,
    max_depth=25,
    min_samples_leaf=25,
    random_state=42,
    n_jobs=-1,
)

start_time = time.time()

rf_run_3.fit(
    X_train_cat,
    y_train_cat
)

rf_run_3_time = time.time() - start_time

rf_run_3_predictions = rf_run_3.predict(
    X_validation_np
)

rf_run_3_mae = mean_absolute_error(
    y_validation_np,
    rf_run_3_predictions
)

rf_run_3_rmse = root_mean_squared_error(
    y_validation_np,
    rf_run_3_predictions
)

rf_run_3_r2 = r2_score(
    y_validation_np,
    rf_run_3_predictions
)

print("Random Forest - Run 3")
print("Training rows = 5,000,000")
print("n_estimators = 150, max_depth = 25, min_samples_leaf = 25")
print("MAE:", round(rf_run_3_mae, 4))
print("RMSE:", round(rf_run_3_rmse, 4))
print("R²:", round(rf_run_3_r2, 4))
print("Training time (seconds):", round(rf_run_3_time, 2))

Random Forest - Run 3
Training rows = 5,000,000
n_estimators = 150, max_depth = 25, min_samples_leaf = 25
MAE: 4.8473
RMSE: 9.3676
R²: 0.7151
Training time (seconds): 1531.95


In [34]:
[name for name in globals() if "tree" in name.lower() or "decision" in name.lower()]

[]